<a href="https://colab.research.google.com/github/Suvanga/ML_DL/blob/main/CNN_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn

import pandas as pd
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
from timeit import default_timer as timer
from torch.utils.data import DataLoader
import requests
from pathlib import Path

import requests
from pathlib import Path

import matplotlib.pyplot as plt

In [ ]:
import requests
from pathlib import Path

if Path("helper_functions.py").is_file():
  print("helper_functoins already exist")
else:
  requests = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py")
  with open("helper_functions.py","wb")as f: ##this sets the permissipon of the file to write binary functions here in the file
    f.write(requests.content)  #now we write the contents inside the helper function.py file we just created

from helper_functions import accuracy_fn




In [ ]:
#device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
##Training Data
train_data = datasets.FashionMNIST(
  root = "data", #Where to download the data to ?
  train = True, #get training data
  download = True, #download data if it doesn't exist
  transform=ToTensor(), #convert to tensor
  target_transform=None
)

#Testing Data
test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = ToTensor(),
    target_transform=None
)

##See the first training example
image , label = train_data[0]
# image , label

class_names = train_data.classes
class_names_idx = train_data.class_to_idx
print (f"\n{class_names_idx}")

In [ ]:
BATCH_SIZE = 32

train_dataloader = DataLoader(dataset = train_data,
                              batch_size = BATCH_SIZE,
                              shuffle = True)
test_dataloader = DataLoader(dataset = test_data,
                              batch_size = BATCH_SIZE,
                              shuffle = False)


from PIL.Image import ImageTransformHandler
##CNN
* It is known for finding pattern in visual data
* Input layer -> Just like any neural network
* Convolution Layer: learns the most important features from the target ImageTransformHandler
* https://poloclub.github.io/cnn-explainer/

In [ ]:
from torch.nn.modules.conv import Conv2d
class FashionMNISTModelV2(nn.Module):
  """
  Model architechture that replicates the tinyVGG
  Model from CNN explainer website is being implemented
  """
  def __init__(self, input_shape:int, hidden_units:int, output_shape: int):
    super() .__init__()
    self.conv_block_1 = nn.Sequential(
        #create a conv layer https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
        nn.Conv2d(in_channels = input_shape,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride = 1,
                  padding = 1
    ),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride =1,
                  padding =1
        ),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size =2,)
    )

    self.conv_block_2 =nn.Sequential(nn.Conv2d(
                                  in_channels = hidden_units,
                                  out_channels = hidden_units,
                                  kernel_size = 3,
                                  stride =1,
                                  padding =1),
        nn.ReLU(),
        nn.Conv2d(in_channels = hidden_units,
                  out_channels = hidden_units,
                  kernel_size = 3,
                  stride =1,
                  padding =1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size =2)
    )
    self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = hidden_units*7*7, # Corrected: Calculate based on image dimensions after conv/pool layers
                      out_features = output_shape)
        )

  def forward(self,x:torch.Tensor):
    x =  self.conv_block_1(x)
    # print(f"Output Shape of Comnv block1{x.shape}")
    x = self.conv_block_2(x)
    # print(f"Output Shape of conv block2 {x.shape}")
    x = self.classifier(x)
    # print(f"Output shape of the classifier {x.shape}")
    return x

In [ ]:
#instantiating a model
torch.manual_seed(42)
model_2 = FashionMNISTModelV2(input_shape = 1,
                              hidden_units = 10,
                              output_shape = len(class_names)).to(device)

# Setting up the optimizer *after* model is instantiated
optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.1)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
rand_image_tensor = torch.randn(size =(1,28,28)).unsqueeze(0).to(device)
rand_image_tensor.shape

In [ ]:
model_2(rand_image_tensor)

#7.1 stepping through `nn.conv2d`

In [ ]:
#creating a batch of images
images = torch.randn(size = (32,3,64,64))
test_image = images[0]

print(f"Image batch shape : {images.shape}")
print(f"Single image shape : {test_image.shape}")
print(f"test image \n {test_image}")


In [ ]:
# code for testing conv_layer for block cell 12 and 13
torch.manual_seed(42)

conv_layer = nn.Conv2d(in_channels = 3,
                       out_channels=10,
                       kernel_size=(3,3),
                       stride =1,
                       padding =0)
conv_output = conv_layer(test_image)
conv_output.shape

In [ ]:
#going through conv2d layer
#printing out original image shape
print(f"Image batch shape : {images.shape}")
print(f"Single image shape : {test_image.shape}")

#creating a sample nn.MaxPool2d layer
max_pool_layer = nn.MaxPool2d(kernel_size =2)

#pass the conv later first
test_image_through_conv =conv_layer(test_image.unsqueeze(dim=0))
print(f"Shape after going through conv layer {test_image_through_conv.shape}")

#Pass the data through max pool layer
max_pool_output = max_pool_layer(test_image_through_conv)
print(f"Shape after going through max pool layer {max_pool_output.shape}")

In [ ]:
#going through max pool layer
torch.manual_seed(42)

random_tensor = torch.randn(size =(1,1,2,2))
random_tensor
print(f"random tensor: {random_tensor}")
print(f"random tensor shape: {random_tensor.shape}\n")

#creating a max pool layer
max_pool_layer = nn.MaxPool2d(kernel_size =2)
max_pool_tensor = max_pool_layer(random_tensor)

#printing the new max pooled tnesor
print(f" max_pool_tensor{max_pool_tensor}")
print("Max pool tensor shape",max_pool_tensor.shape)



##Trainign and testing our first CNN
* Setup loss and optimizer


In [ ]:
#instantiating a model
torch.manual_seed(42)
model_2 = FashionMNISTModelV2(input_shape = 1,
                              hidden_units = 10,
                              output_shape = len(class_names)).to(device)

# Setting up the optimizer *after* model is instantiated
optimizer = torch.optim.SGD(params=model_2.parameters(), lr=0.1)
loss_fn = nn.CrossEntropyLoss()

In [ ]:

import torch
from tqdm.auto import tqdm

def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               accuracy_fn,
               device: torch.device):
    train_loss, train_acc = 0, 0
    model.train()
    for batch, (X, y) in enumerate(data_loader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss
        train_acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(data_loader) # Changed from /= to =
    train_acc /= len(data_loader)
    print(f"Train loss: {train_loss:.5f} | Train acc: {train_acc:.2f}%")

def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              accuracy_fn,
              device: torch.device):
    test_loss, test_acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device), y.to(device)
            test_pred = model(X)
            test_loss += loss_fn(test_pred, y)
            test_acc += accuracy_fn(y_true=y, y_pred=test_pred.argmax(dim=1))
    test_loss = test_loss / len(data_loader) # Changed from /= to =
    test_acc /= len(data_loader)
    print(f"Test loss: {test_loss:.5f} | Test acc: {test_acc:.2f}%")

def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn,
               device: torch.device):
    loss, acc = 0, 0
    model.eval()
    with torch.inference_mode():
        for X, y in tqdm(data_loader):
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            loss += loss_fn(y_pred, y)
            acc += accuracy_fn(y_true=y, y_pred=y_pred.argmax(dim=1))
        loss = loss / len(data_loader) # Changed from /= to =
        acc /= len(data_loader)
    return {"model_name": model.__class__.__name__,
            "model_loss": loss.item(),
            "model_acc": acc}


In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
from tqdm.auto import tqdm
#Measure
train_time_start_model_2 = timer()

#Train and test the model
epochs = 3

for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n---------")
  train_step(model = model_2,
             data_loader = train_dataloader,
             optimizer = optimizer,
             loss_fn = loss_fn,
             accuracy_fn = accuracy_fn,
             device = device)

  test_step(model = model_2,
            data_loader = test_dataloader,
            loss_fn = loss_fn,
            accuracy_fn = accuracy_fn, # Added accuracy_fn
            device = device)

train_time_end_model_2 = timer()
total_train_time_model_2 = train_time_end_model_2 - train_time_start_model_2
print(f"Total training time: {total_train_time_model_2:.3f} seconds")

In [ ]:
model_2_results = eval_model(model = model_2,
                             data_loader = test_dataloader,
                             loss_fn = loss_fn,
                             accuracy_fn = accuracy_fn,
                             device = device)
model_2_results

In [ ]:
# Manually inputting results from your other files
model_0_results = {
    "model_name": "FashionMNISTModelV0",
    "model_loss": 0.476,
    "model_acc": 83.42,
    "training_time": 45.12
}

model_1_results = {
    "model_name": "FashionMNISTModelV1",
    "model_loss": 0.685,
    "model_acc": 75.01,
    "training_time": 150.34
}

# Add training time to your current model results
model_2_results["training_time"] = 182.089


In [ ]:
import pandas as pd

# Combine all results into a list
all_model_results = [model_0_results, model_1_results, model_2_results]

# Create DataFrame
compare_df = pd.DataFrame(all_model_results)

# Calculate accuracy per second to see efficiency
compare_df["acc_per_second"] = compare_df["model_acc"] / compare_df["training_time"]

print(compare_df)

In [ ]:
compare_df.set_index("model_name")["model_acc"].plot(kind="barh")
plt.xlabel("accuracy(%)")
plt.ylabel("model")

In [ ]:
def make_predictions(model:torch.nn.Module,
                     data:list,
                     device: torch.device =device):
  all_pred_probs =[]
  model.to(device)
  model.eval()

  with torch.inference_mode():
    for sample in data :
      sample = torch.unsqueeze(sample, dim =0).to(device)
      pred_logits = model(sample)

      pred_probs_tensor = torch.softmax(pred_logits.squeeze(), dim =0)

      all_pred_probs.append(pred_probs_tensor.cpu()) # Append tensor directly, without converting to numpy

  return torch.stack(all_pred_probs) # Stack the list of tensors into a single tensor

In [ ]:
import random
# random.seed(42)
test_samples = []
test_labels = []
for sample, label in random.sample(list(test_data), k=9):
    test_samples.append(sample)
    test_labels.append(label)

# View the first sample shape
test_samples[0].shape

In [ ]:
plt.imshow(test_samples[0].squeeze(), cmap="gray")
plt.title(class_names[test_labels[0]])

In [ ]:
pred_probs = make_predictions(model = model_2,
                             data = test_samples)

In [ ]:
pred_probs[:2]

In [ ]:
pred_classes = torch.argmax(pred_probs, dim =1)
pred_classes

In [ ]:
test_labels

In [ ]:
plt.figure(figsize =(9,9))
nrows = 3
ncols = 3

for i , sample in enumerate(test_samples):
  #create subplot
  plt.subplot(nrows, ncols, i+1)

  #plot tehe target image
  plt.imshow(sample.squeeze(), cmap ="gray")

  #find the preddiction
  pred_label = class_names[pred_classes[i]]

  #get the truth label in text form
  true_label = class_names[test_labels[i]]

  #
  title_text = f"Pred: {pred_label} | Truth: {true_label}"
  if pred_label == true_label:
    plt.title(title_text, fontsize =10, color ="green") #green text if the prediction is same as the truth

  else:
    plt.title(title_text, fontsize =10, color ="red") #red text
  plt.axis = False

In [ ]:
def make_predictions(model:torch.nn.Module,
                     data:list,
                     device: torch.device =device):
  all_pred_probs =[]
  model.to(device)
  model.eval()

  with torch.inference_mode():
    for sample in data :
      sample = torch.unsqueeze(sample, dim =0).to(device)
      pred_logits = model(sample)

      pred_probs_tensor = torch.softmax(pred_logits.squeeze(), dim =0)

      all_pred_probs.append(pred_probs_tensor.cpu()) # Append tensor directly, without converting to numpy

  return torch.stack(all_pred_probs) # Stack the list of tensors into a single tensor

In [ ]:
#making a confusion matrix
from tqdm.auto import tqdm
y_preds = []
model_2.eval()

with torch.inference_mode():
  for X, y in tqdm(test_dataloader, desc = "Making predictions ..."):
    X, y = X.to(device), y.to(device)

    y_logit = model_2(X)

    y_pred = torch.softmax(y_logit.squeeze(), dim =0).argmax(dim =1)

    y_preds.append(y_pred.cpu())

  #
  # print(y_preds)
  y_preds_tensor = torch.cat(y_preds)
  y_preds_tensor






In [ ]:
len(y_preds_tensor)

In [ ]:
import mlxtend

In [ ]:
try:
  import torchmetrics, mlxtend
  print(f"mlxtend version : {mlxtend.__version__}")
  assert int(mlxtend.__version__.split(".")[1])>=19, "should be higher than 0.19"
except :
  !pip install -U mlxtend torchmetrics

In [ ]:
  import mlxtend
  print(f"mlxtend version : {mlxtend.__version__}")


In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

confmat = ConfusionMatrix(task='multiclass', num_classes = len(class_names))
confmat_tensor = confmat(preds = y_preds_tensor, target = test_data.targets)
confmat_tensor

fig , ax = plot_confusion_matrix(conf_mat = confmat_tensor.numpy(),
                                 class_names = class_names,
                                 figsize = (10,10))
plt.show()

In [ ]:
#save and load the best performing model
from pathlib import Path

# Create model directory path
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True,
                 exist_ok=True)

# Create model save path
MODEL_NAME = "03_pytorch_computer_vision_model_2.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# save the model
print(f"Saving the model to {MODEL_SAVE_PATH}")
torch.save(obj = model_2.state_dict(),
           f = MODEL_SAVE_PATH)

In [ ]:
model_2_results

In [ ]:
#creating a new instance
torch.manual_seed(42)

loaded_model_2 = FashionMNISTModelV2(input_shape  = 1,
                                    hidden_units = 10,
                                    output_shape = len(class_names)).to(device)
loaded_model_2.load_state_dict(torch.load(MODEL_SAVE_PATH))



In [ ]:
loaded_model_2_results = eval_model(
    model = loaded_model_2,
    data_loader = test_dataloader,
    loss_fn = loss_fn,
    accuracy_fn = accuracy_fn,
    device = device)

In [ ]:
torch.isclose(torch.tensor(model_2_results["model_loss"]),
              torch.tensor(loaded_model_2_results["model_loss"]))
